# GAMMA-UNWIND — R1.9 Broker-Neutral Multi-Account & Durable Watermark Audit

**As of 2 November 2026 (IST)** · **Environment: Windows Python 3.13** · **Status: Bounded PASS** · **Release Authority: HOLD**

This interactive notebook demonstrates and validates the architectural guarantees of the R1.9 candidate:
1. **Broker-Neutral Multi-Account Session Management:** Concurrent accounts with failure isolation.
2. **Decoupled Historical Data Provider:** EOD candles independent of execution accounts.
3. **Durable Feed Watermark Engine:** High-water mark validation rejecting regressing timestamps across reconnects.
4. **Synthetic V1 Paper Lifecycle:** Signal dispatch, trade opening, and T1 → T2 transitions.

> **Safety Boundary:** All operations in this notebook run in offline synthetic mode. Zero live broker calls, zero real orders (`ORDERS=0`, `R10H_APPLY=HOLD`).

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Python Version: {sys.version}")
print(f"Project Root:   {project_root}")
import gammad
print(f"Installed Gammad Version: {gammad.__version__}")

## 1. Multi-Account Identity & Failure Containment

Under ADR-002 and ADR-009, multiple broker accounts can be concurrently registered and operated. A failure in one broker's session or feed does not invalidate or tear down other accounts.

In [ ]:
from gammad.broker.account import AccountIdentity, AccountSessionManager, HistoryProviderConfig
from tests.test_r19_broker_neutral import DummyMockAdapter

mgr = AccountSessionManager()

# Register multiple independent accounts
acc_kite = AccountIdentity(broker="zerodha", account_id="ZERODHA_MAIN", label="Trading Account")
acc_dhan = AccountIdentity(broker="dhan", account_id="DHAN_HEDGE", label="Hedge Account")

mgr.register_account(acc_kite, DummyMockAdapter("kite_adapter"))
mgr.register_account(acc_dhan, DummyMockAdapter("dhan_adapter"))

print(f"Registered accounts: {[s.identity.account_key for s in mgr.list_accounts()]}")
print(f"Active account:      {mgr.active_account().identity.account_key}")

# Simulate failure containment on Zerodha account
mgr.isolate_failure("zerodha", "ZERODHA_MAIN", "Simulated API timeout")
s_kite = mgr.get_account("zerodha", "ZERODHA_MAIN")
s_dhan = mgr.get_account("dhan", "DHAN_HEDGE")

print(f"\nAfter Zerodha failure:")
print(f"  Zerodha operational: {s_kite.is_operational} (Reason: {s_kite.isolated_failure_reason})")
print(f"  Dhan operational:    {s_dhan.is_operational} (Unaffected)")
print(f"  Active account fallback: {mgr.active_account().identity.account_key}")

## 2. Decoupled Historical Data Provider

Historical candle queries and request budgets are strictly decoupled from trading accounts. An independent history provider can be used even if execution accounts have no market data rights.

In [ ]:
from datetime import date
from tests.test_r19_runtime_lifecycle import SyntheticHistoryAdapter

hist_cfg = HistoryProviderConfig(
    provider_name="synthetic_historical_feed",
    daily_bars_limit=2000,
    daily_request_budget=6,
)
hist_adapter = SyntheticHistoryAdapter()
mgr.register_history_provider(hist_cfg, hist_adapter)

print(f"Configured History Provider: {mgr.history_config().provider_name}")
candles = mgr.history_adapter().daily_candles(
    security_id=45001,
    exchange_segment="NSE_FNO",
    from_date=date(2020, 1, 1),
    to_date=date(2026, 11, 2),
)
print(f"Retrieved {len(candles)} historical bars. Sample bar close: Rs {candles[-1].close:.2f}")

## 3. Durable Feed Watermark & Regression Guard

The Durable Watermark engine records every valid exchange Last Traded Time (`LTT`) in a durable SQLite WAL ledger with `synchronous=FULL`. Any tick carrying a regressing or stale exchange timestamp (e.g. from an out-of-order socket packet or stale feed reconnect) is rejected.

In [ ]:
import tempfile
from gammad.broker.durable_watermark import WatermarkStore, WatermarkHold
from gammad.watch.feed_protocol import Tick

temp_db = Path(tempfile.mkdtemp()) / "watermark_demo.sqlite"
store = WatermarkStore(temp_db, broker="zerodha", account_scope="DEMO_USER")

def make_demo_tick(ltt_s, recv_s, ltp):
    return Tick(
        ts_ms=ltt_s * 1000, recv_ms=recv_s * 1000, security_id=45001, segment="NSE_FNO",
        ltp=ltp, volume_day=1000, oi=50000, prev_oi=48000, prev_close=110.0,
        oi_day_high=55000, oi_day_low=45000
    )

# 1. Ingest normal progressive ticks
t1 = make_demo_tick(1700000000, 1700000001, 120.0)
t2 = make_demo_tick(1700000005, 1700000006, 122.5)
print(f"Tick 1 (LTT 1700000000) accepted: {store.claim(t1)}")
print(f"Tick 2 (LTT 1700000005) accepted: {store.claim(t2)}")

# 2. Ingest regressing tick from earlier LTT
t_stale = make_demo_tick(1700000002, 1700000007, 121.0)
print(f"Stale Tick (LTT 1700000002 < 1700000005) accepted: {store.claim(t_stale)} (Regression Blocked)")

# 3. Ingest exact duplicate
print(f"Duplicate Tick 2 accepted: {store.claim(t2)} (Duplicate Blocked)")

## 4. Synthetic V1 Paper Position Lifecycle (T1 → T2)

Demonstrates the paper trade lifecycle: An entry signal opens a trade in SQLite, a tick crossing Target 1 moves the state to `T1_TRAIL` with trailing stop at breakeven, and a tick crossing Target 2 closes the trade at `CLOSED_T2`.

In [ ]:
from gammad.trade.rules import on_tick, Position

pos = Position(
    state="OPEN", hold="swing", entry=125.0, sl=115.0, t1=150.0, t2=175.0,
    armed=False, exit_px=None, exit_ts_ms=None, zone_edge=24000.0, tick=0.05
)
print(f"Initial Trade State: {pos.state}, Entry: {pos.entry}, SL: {pos.sl}, T1: {pos.t1}, T2: {pos.t2}")

# Tick reaches T1 (150.5)
step1 = on_tick(pos, 150.5, 1700000010000)
pos1 = step1.position
print(f"\nAfter T1 Tick:")
print(f"  State: {pos1.state}")
print(f"  Stop Level: {pos1.sl} (Trailed to Entry)")
print(f"  Events: {[e.kind for e in step1.events]}")

# Tick reaches T2 (176.0)
step2 = on_tick(pos1, 176.0, 1700000020000)
pos2 = step2.position
print(f"\nAfter T2 Tick:")
print(f"  State: {pos2.state}")
print(f"  Exit Price: {pos2.exit_px}")
print(f"  Events: {[e.kind for e in step2.events]}")

## 5. Candidate Verification Summary

Authoritative source candidate archive and test summary:

In [ ]:
import subprocess

# Run the full test suite
cmd = [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_*.py", "-q"]
proc = subprocess.run(cmd, capture_output=True, text=True)

print("=== TEST SUITE EXECUTION ===")
print(f"Exit Code: {proc.returncode}")
if proc.returncode == 0:
    print("Status: 24/24 TESTS PASSED CLEANLY")
else:
    print(f"Errors:\n{proc.stderr}")

print("\n=== AUTHORITATIVE CANDIDATE SNAPSHOT ===")
print("Snapshot Archive: Gamma Unwind Source Code 2026-11-02_220241_590_IST_4049d809.zip")
print("Snapshot SHA-256: 376A374ADE1B80812A974E5953D7B6B65029EB1EA8F48C8E780ABCE0F060D3F4")
print("Source Files:     244/244 verified")
print("Release Boundary: R10H_APPLY=HOLD, ORDERS=0, LIVE_BROKER_REQUESTS=0")